In [ ]:
import os, csv, json, uuid, time, random
from datetime import datetime, timedelta
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

RUN_ID = str(uuid.uuid4())
DB_SYSTEM = "AstraDB"
SERVER_VERSION = "Cassandra 4.0.11"
DRIVER = "cassandra-driver-3.x"
CONNECTION_PARAMS = "Astra secure bundle"
DB_NAME = "research-benchmarking"

os.makedirs("./logs", exist_ok=True)

# auth = PlainTextAuthProvider('token')
# cluster = Cluster(cloud={'secure_connect_bundle': 'C:/Users/avyaa/bundle.zip'}, auth_provider=auth)
# session = cluster.connect('ecommerce')

print("Connected, RUN_ID:", RUN_ID)

Connected, RUN_ID: bd35ad70-ec40-453e-a523-c789c1bb370e


In [2]:
def exec_and_time(prepared, params=(), timeout=None):
    t0 = time.perf_counter()
    try:
        rs = session.execute(prepared, params, timeout=timeout)
        rows = list(rs)
        return True, (time.perf_counter()-t0)*1000.0, rows, None, None
    except Exception as e:
        return False, (time.perf_counter()-t0)*1000.0, [], type(e).__name__, str(e)

def sample_ids():
    p = list(session.execute("SELECT id, sku, category_id, name_lc FROM products LIMIT 200"))
    c = list(session.execute("SELECT id FROM customers LIMIT 200"))
    o = list(session.execute("SELECT id, order_date FROM orders LIMIT 500"))
    return {
        "product_ids": [r.id for r in p],
        "product_skus": [r.sku for r in p],
        "category_ids": list({r.category_id for r in p}),
        "customer_ids": [r.id for r in c],
        "order_ids": [r.id for r in o],
    }

SAMPLE = sample_ids()
print({k: len(v) for k,v in SAMPLE.items()})

{'product_ids': 200, 'product_skus': 200, 'category_ids': 48, 'customer_ids': 200, 'order_ids': 500}


In [3]:
from collections import namedtuple
Q = namedtuple("Q", "id name complexity cql param_gen kind")
import uuid, random
from datetime import datetime, timedelta

# BASIC
R1 = Q("R1","Product by ID","Basic",
       "SELECT id,sku,name,price,category_id FROM products WHERE id = ?",
       lambda: (uuid.UUID(str(random.choice(SAMPLE['product_ids']))),),
       "single")

R2 = Q("R2","Product by SKU","Basic",
       "SELECT id,sku,name,price,category_id FROM products WHERE sku_lc = ?",
       lambda: (str(random.choice(SAMPLE['product_skus'])).lower(),),
       "single")

R3 = Q("R3","Customer order history","Basic",
       "SELECT id,order_date,total_amount FROM orders WHERE customer_id = ? LIMIT 50",
       lambda: (uuid.UUID(str(random.choice(SAMPLE['customer_ids']))),),
       "single")

R4 = Q("R4","Category browse","Basic",
       "SELECT id,name,price FROM products WHERE category_id = ? LIMIT 20",
       lambda: (uuid.UUID(str(random.choice(SAMPLE['category_ids']))),),
       "single")

# MODERATE (requires SAI on name_lc, category_id, price if price range is used)
# Choose an existing name_lc to ensure hits
names_seed = [row.name_lc for row in session.execute("SELECT name_lc FROM products LIMIT 200") if getattr(row,'name_lc',None)]
def pick_existing_name(): return random.choice(names_seed) if names_seed else "n/a"

R5 = Q("R5","Search with filters (equality + range)","Moderate",
       "SELECT id,name,price FROM products WHERE name_lc = ? AND category_id = ? AND price >= ? AND price <= ? LIMIT 200",
       lambda: (pick_existing_name(),
                uuid.UUID(str(random.choice(SAMPLE['category_ids']))),
                random.choice([10.0,50.0,100.0]),
                random.choice([200.0,500.0,1000.0])),
       "single")

# Best sellers (client-side composition; no server JOIN)
R6a = Q("R6a","Recent order IDs","Moderate",
        "SELECT id FROM orders WHERE order_date >= ? LIMIT 2000",
        lambda: (datetime.utcnow() - timedelta(days=30),),
        "single")
R6b = Q("R6b","Order items by order_id","Moderate",
        "SELECT product_id,quantity FROM order_items WHERE order_id = ?",
        None,"prepared_only")

# New arrivals (requires SAI on created_at)
R7 = Q("R7","New arrivals by window","Moderate",
       "SELECT id,name,created_at FROM products WHERE category_id = ? AND created_at >= ? LIMIT 10",
       lambda: (uuid.UUID(str(random.choice(SAMPLE['category_ids']))),
                datetime.utcnow() - timedelta(days=60)),
       "single")

# Order detail with items (two-step)
R8a = Q("R8a","Order header","Moderate",
        "SELECT id,order_date,total_amount FROM orders WHERE id = ?",
        lambda: (uuid.UUID(str(random.choice(SAMPLE['order_ids']))),),
        "single")
R8b = Q("R8b","Order lines","Moderate",
        "SELECT product_id,quantity,unit_price FROM order_items WHERE order_id = ?",
        None,"prepared_only")

# ADVANCED (client-side aggregation; requires SAI on price for the window)
R9 = Q("R9","Brand facet count (client-side)","Advanced",
       "SELECT brand FROM products WHERE category_id = ? AND price >= ? AND price <= ? LIMIT 5000",
       lambda: (uuid.UUID(str(random.choice(SAMPLE['category_ids']))), 50.0, 500.0),
       "single")

R10 = Q("R10","Attribute-like filter (client-side sort)","Advanced",
        "SELECT id,name,price,popularity,attributes FROM products WHERE category_id = ? AND price >= ? AND price <= ? LIMIT 1000",
        lambda: (uuid.UUID(str(random.choice(SAMPLE['category_ids']))),
                 random.choice([10.0,50.0,100.0]),
                 random.choice([200.0,500.0,1000.0])),
        "single")

to_prepare = [R1,R2,R3,R4,R5,R6a,R6b,R7,R8a,R8b,R9,R10]
PREP = {q.id: session.prepare(q.cql) for q in to_prepare}
print("Prepared:", list(PREP.keys()))

Prepared: ['R1', 'R2', 'R3', 'R4', 'R5', 'R6a', 'R6b', 'R7', 'R8a', 'R8b', 'R9', 'R10']


In [4]:
# Run this to verify Cell 3 worked:
print("PREP keys:", list(PREP.keys()) if 'PREP' in globals() else "PREP not found")
print("META ready:", 'META' in globals())

PREP keys: ['R1', 'R2', 'R3', 'R4', 'R5', 'R6a', 'R6b', 'R7', 'R8a', 'R8b', 'R9', 'R10']
META ready: False


In [5]:
from collections import defaultdict
from tqdm import tqdm
import json, csv, time

# Missing helper functions
def log_row(writer, meta, qid, qname, complexity, run_no, is_cold, params, ok, ms, rowcount, errc, errm, mode, psrc):
    writer.writerow({
        "timestamp_utc": datetime.utcnow().isoformat(),
        "run_id": meta["run_id"],
        "db_system": meta["db_system"],
        "server_version": meta["server_version"],
        "driver": meta["driver"],
        "connection_params": meta["connection_params"],
        "db_name": meta["db_name"],
        "mode": mode,
        "param_source": psrc,
        "query_id": qid,
        "complexity": complexity,
        "query_name": qname,
        "operation_type": "read",
        "run_number": run_no,
        "is_cold": is_cold,
        "execution_ms": round(ms,3) if ok else None,
        "row_count": rowcount if ok else None,
        "error_code": errc,
        "error_message": errm,
        "params_json": json.dumps(params, default=str, separators=(",",":"))
    })

# Missing META object
META = {
    "run_id": RUN_ID,
    "db_system": DB_SYSTEM,
    "server_version": SERVER_VERSION,
    "driver": DRIVER,
    "connection_params": CONNECTION_PARAMS,
    "db_name": DB_NAME
}

# Missing composite query runners
def run_R6_best_sellers():
    cutoff = (datetime.utcnow() - timedelta(days=30),)
    ok1, ms1, rows1, e1c, e1m = exec_and_time(PREP["R6a"], cutoff, timeout=30)
    if not ok1:
        return False, ms1, [], e1c, e1m
    
    from collections import Counter
    c = Counter()
    t0 = time.perf_counter()
    for r in rows1[:2000]:
        ok2, ms2, rows2, e2c, e2m = exec_and_time(PREP["R6b"], (r.id,), timeout=10)
        if not ok2:
            continue
        for li in rows2:
            c[li.product_id] += int(li.quantity)
    dt = (time.perf_counter() - t0) * 1000.0
    top10 = c.most_common(10)
    return True, ms1+dt, top10, None, None

def run_R8_order_detail():
    oid = uuid.UUID(str(random.choice(SAMPLE["order_ids"])))
    ok1, ms1, rows1, e1c, e1m = exec_and_time(PREP["R8a"], (oid,), timeout=20)
    if not ok1:
        return False, ms1, [], e1c, e1m
    ok2, ms2, rows2, e2c, e2m = exec_and_time(PREP["R8b"], (oid,), timeout=20)
    if not ok2:
        return False, ms1+ms2, [], e2c, e2m
    return True, ms1+ms2, rows2, None, None

print("✅ Helper functions defined!")

✅ Helper functions defined!


In [6]:
# A1: fixed parameters, 1 cold + 10 warm per query, interleaved sequence with composites (R6, R8)
from collections import defaultdict
from tqdm import tqdm
import json, csv, time

def run_A1_interleaved(outfile="./logs/reads_A1.csv"):
    # Build query dictionary and available IDs from prepared statements
    QD = {q.id: q for q in [R1,R2,R3,R4,R5,R6a,R6b,R7,R8a,R8b,R9,R10]}
    single_ids_all = ["R1","R2","R3","R4","R5","R7","R9","R10"]
    single_ids = [qid for qid in single_ids_all if qid in PREP]
    use_R6 = ("R6a" in PREP and "R6b" in PREP)
    use_R8 = ("R8a" in PREP and "R8b" in PREP)
    
    # Fixed parameter set for A1
    fixed_params = {}
    for qid in single_ids:
        fixed_params[qid] = QD[qid].param_gen()
    
    # CSV header
    fields = ["timestamp_utc","run_id","db_system","server_version","driver","connection_params","db_name",
              "mode","param_source","query_id","complexity","query_name","operation_type",
              "run_number","is_cold","execution_ms","row_count","error_code","error_message","params_json"]
    # Overwrite file each run
    f = open(outfile, "w", newline="", encoding="utf-8")
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    
    # Progress and error suppression after first failure per query
    disabled = set()
    success_count = defaultdict(int)
    error_count = defaultdict(int)
    
    total_iters = 11 * (len(single_ids) + (1 if use_R6 else 0) + (1 if use_R8 else 0))
    pbar = tqdm(total=total_iters, desc="A1 (interleaved)", unit="ops")
    
    for i in range(1, 12):
        is_cold = (i == 1)
        
        # Interleave R1..R4..R10 each loop
        for qid in single_ids:
            if qid in disabled:
                pbar.update(1)
                continue
            q = QD[qid]
            params = fixed_params[qid]
            ok, ms, rows, ec, em = exec_and_time(PREP[qid], params, timeout=30)
            log_row(w, META, qid, q.name, q.complexity, i, is_cold, params, ok, ms, (len(rows) if ok else None), ec, em, "A1", "fixed")
            if ok:
                success_count[qid] += 1
            else:
                error_count[qid] += 1
                # If the cold run fails with InvalidRequest, disable to save time
                if is_cold and ec == "InvalidRequest":
                    disabled.add(qid)
            pbar.update(1)
        
        # Composite R6 (best sellers) after each loop if available
        if use_R6:
            ok, ms, rows, ec, em = run_R6_best_sellers()
            params = {"cutoff_days": 30}
            log_row(w, META, "R6", "Best Sellers 30 Day", "Moderate", i, is_cold, params, ok, ms, (len(rows) if ok else None), ec, em, "A1", "fixed")
            if ok:
                success_count["R6"] += 1
            else:
                error_count["R6"] += 1
            pbar.update(1)
        
        # Composite R8 (order detail) after each loop if available
        if use_R8:
            ok, ms, rows, ec, em = run_R8_order_detail()
            params = {}
            log_row(w, META, "R8", "Order detail with items", "Moderate", i, is_cold, params, ok, ms, (len(rows) if ok else None), ec, em, "A1", "fixed")
            if ok:
                success_count["R8"] += 1
            else:
                error_count["R8"] += 1
            pbar.update(1)
    
    pbar.close()
    f.flush(); f.close()
    print("A1 complete →", outfile)
    print("Success summary:", dict(success_count))
    print("Error summary:", dict(error_count))

In [7]:
# A2: random parameters per execution, interleaved sequence with composites (R6, R8)
from collections import defaultdict
from tqdm import tqdm
import json, csv, time

def run_A2_interleaved(outfile="./logs/reads_A2.csv"):
    QD = {q.id: q for q in [R1,R2,R3,R4,R5,R6a,R6b,R7,R8a,R8b,R9,R10]}
    single_ids_all = ["R1","R2","R3","R4","R5","R7","R9","R10"]
    single_ids = [qid for qid in single_ids_all if qid in PREP]
    use_R6 = ("R6a" in PREP and "R6b" in PREP)
    use_R8 = ("R8a" in PREP and "R8b" in PREP)
    
    fields = ["timestamp_utc","run_id","db_system","server_version","driver","connection_params","db_name",
              "mode","param_source","query_id","complexity","query_name","operation_type",
              "run_number","is_cold","execution_ms","row_count","error_code","error_message","params_json"]
    f = open(outfile, "w", newline="", encoding="utf-8")
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    
    disabled = set()
    success_count = defaultdict(int)
    error_count = defaultdict(int)
    
    total_iters = 10 * (len(single_ids) + (1 if use_R6 else 0) + (1 if use_R8 else 0))
    pbar = tqdm(total=total_iters, desc="A2 (interleaved)", unit="ops")
    
    for i in range(1, 11):
        # Interleave R1..R10 each loop with fresh random params
        for qid in single_ids:
            if qid in disabled:
                pbar.update(1)
                continue
            q = QD[qid]
            params = q.param_gen()
            ok, ms, rows, ec, em = exec_and_time(PREP[qid], params, timeout=30)
            log_row(w, META, qid, q.name, q.complexity, i, False, params, ok, ms, (len(rows) if ok else None), ec, em, "A2", "random")
            if ok:
                success_count[qid] += 1
            else:
                error_count[qid] += 1
                # If first failure is InvalidRequest, disable for the rest
                if i == 1 and ec == "InvalidRequest":
                    disabled.add(qid)
            pbar.update(1)
        
        if use_R6:
            ok, ms, rows, ec, em = run_R6_best_sellers()
            params = {"cutoff_days": 30}
            log_row(w, META, "R6", "Best Sellers 30 Day", "Moderate", i, False, params, ok, ms, (len(rows) if ok else None), ec, em, "A2", "random")
            if ok:
                success_count["R6"] += 1
            else:
                error_count["R6"] += 1
            pbar.update(1)
        
        if use_R8:
            ok, ms, rows, ec, em = run_R8_order_detail()
            params = {}
            log_row(w, META, "R8", "Order detail with items", "Moderate", i, False, params, ok, ms, (len(rows) if ok else None), ec, em, "A2", "random")
            if ok:
                success_count["R8"] += 1
            else:
                error_count["R8"] += 1
            pbar.update(1)
    
    pbar.close()
    f.flush(); f.close()
    print("A2 complete →", outfile)
    print("Success summary:", dict(success_count))
    print("Error summary:", dict(error_count))

In [8]:
print("🔥 Starting AstraDB Read Benchmarking")
print("=" * 50)

# Run A1 (Fixed Parameters)
print("\n📊 Phase A1: Fixed Parameters (Cold + Warm)")
print("Sequence: R1→R2→R3→R4→R5→R7→R9→R10→R6→R8 per iteration")
print("Runs: 1 cold + 10 warm per query (11 total per query)")

start_time = time.time()
run_A1_interleaved("./logs/reads_A1.csv")
a1_duration = time.time() - start_time

print(f"⏱️  A1 completed in {a1_duration:.1f} seconds")

# Run A2 (Random Parameters)
print("\n📊 Phase A2: Random Parameters")
print("Sequence: R1→R2→R3→R4→R5→R7→R9→R10→R6→R8 per iteration")
print("Runs: 10 runs per query with fresh random parameters")

start_time = time.time()
run_A2_interleaved("./logs/reads_A2.csv")
a2_duration = time.time() - start_time

print(f"⏱️  A2 completed in {a2_duration:.1f} seconds")

# Final Summary
print("\n🎉 ASTRADB READ BENCHMARKING COMPLETE!")
print("=" * 50)
print(f"📁 Files generated:")
print(f"   - ./logs/reads_A1.csv")
print(f"   - ./logs/reads_A2.csv")
print(f"⏱️  Total time: {(a1_duration + a2_duration):.1f} seconds")
print(f"🔬 Run ID: {RUN_ID}")
print(f"🚀 Ready for analysis phase!")

🔥 Starting AstraDB Read Benchmarking

📊 Phase A1: Fixed Parameters (Cold + Warm)
Sequence: R1→R2→R3→R4→R5→R7→R9→R10→R6→R8 per iteration
Runs: 1 cold + 10 warm per query (11 total per query)


A1 (interleaved): 100%|███████████████████████████████████████████████████████████| 110/110 [1:13:55<00:00, 40.32s/ops]


A1 complete → ./logs/reads_A1.csv
Success summary: {'R1': 11, 'R2': 11, 'R3': 11, 'R4': 11, 'R5': 11, 'R7': 11, 'R9': 11, 'R10': 11, 'R6': 11, 'R8': 11}
Error summary: {}
⏱️  A1 completed in 4435.7 seconds

📊 Phase A2: Random Parameters
Sequence: R1→R2→R3→R4→R5→R7→R9→R10→R6→R8 per iteration
Runs: 10 runs per query with fresh random parameters


A2 (interleaved): 100%|███████████████████████████████████████████████████████████| 100/100 [1:04:48<00:00, 38.88s/ops]

A2 complete → ./logs/reads_A2.csv
Success summary: {'R1': 10, 'R2': 10, 'R3': 10, 'R4': 10, 'R5': 10, 'R7': 10, 'R9': 10, 'R10': 10, 'R6': 10, 'R8': 10}
Error summary: {}
⏱️  A2 completed in 3888.3 seconds

🎉 ASTRADB READ BENCHMARKING COMPLETE!
📁 Files generated:
   - ./logs/reads_A1.csv
   - ./logs/reads_A2.csv
⏱️  Total time: 8324.0 seconds
🔬 Run ID: bd35ad70-ec40-453e-a523-c789c1bb370e
🚀 Ready for analysis phase!
